# Which prefix positions carry kappa?

Registered ablation (A1-A4 committed before the run; see README). Same split as the sibling experiments: 2185 stationary census rules train, 162 fresh stationary rules test, disjoint. The 8-letter lookup P8 is re-scored with the key ignoring letter j (masking), and, as a group-count control, with the letters at position j permuted among training rules (200 permutations).

**Assumption.** Same stationarity filter and fresh-rule target as the W3c audit.

In [1]:
import json
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
d = json.loads((root / "experiments/archive/2026-10-kappa-prefix-ablation/results/ablate.json").read_text())
print("inputs:", {k: v[:12] for k, v in d["input_sha256"].items()}, "train", d["n_train"], "test", d["n_test"], "overlap", d["overlap"])
print(f"P8 R^2 {d['r2_base']:+.3f}")
for j, r in d["positions"].items():
    lo, hi = r["loss_ci"]
    print(f"pos {j}: loss {r['loss']:+.3f} [{lo:+.3f}, {hi:+.3f}]   control loss {r['control_loss']:+.3f}")
print(d["verdicts"])
assert d["overlap"] == 0
assert [v for v in d["verdicts"].values()] == [True, True, False, True]  # A3 failed, as in the README
loss = {int(j): r["loss"] for j, r in d["positions"].items()}
assert max(loss, key=loss.get) == 5 and loss[8] < 0.1

inputs: {'probe_summary.csv': 'c76c7927185b', 'prefix_audit_predictions.csv': '742f0fe1d776'} train 2185 test 162 overlap 0
P8 R^2 +0.624
pos 1: loss +0.000 [+0.000, +0.000]   control loss -0.000
pos 2: loss +0.229 [+0.128, +0.331]   control loss +0.252
pos 3: loss +0.255 [+0.138, +0.383]   control loss +0.280
pos 4: loss +0.345 [+0.223, +0.474]   control loss +0.388
pos 5: loss +0.400 [+0.230, +0.565]   control loss +0.442
pos 6: loss +0.294 [+0.171, +0.405]   control loss +0.321
pos 7: loss +0.216 [+0.096, +0.321]   control loss +0.234
pos 8: loss +0.073 [-0.002, +0.157]   control loss +0.099
{'A1 position 1 loss within 0.01 of 0': True, 'A2 positions 2,3,4 each loss >= 0.05': True, 'A3 no position 5-8 loss above max of 2-4': False, 'A4 control loss >= masking loss - 0.05 at every position': True}


## What this shows and does not show
- Information is **not** front-loaded (A3 failed): ablating position 5 costs the most (0.40), positions 2-7 each cost 0.2-0.4, position 8 costs 0.07 with an interval that includes 0. Position 1 is constant in this cohort and costs nothing.
- The permutation control agrees with masking at every position, so this is not a regrouping artifact.
- **Not shown:** whether the signal is additive over positions or needs interactions. The lookup is a conjunctive key, and the losses sum to far more than the 0.62 total, which points to interaction or redundancy but does not separate them.

## Next
A registered additive (letter-by-position) model against the full 8-letter lookup on the same split.